# Bell Pepper Dataset generator

In [ ]:
# imports

import os
from IPython.display import display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import gradio as gr
import torch

from styles import CSS
import csv
import io

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

hf_token = os.getenv('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [ ]:
system_message = """
You are generating synthetic agricultural data for a bell pepper farm.

Generate a CSV dataset with the following columns in this exact order:

Seed_Variant,Disease_Resistance,Fertilizer_Used,Planting_Date,Harvest_Date,Irrigation_Method,Yield_kg

Rules:
- Output ONLY valid CSV.
- Do NOT include explanations, comments, or markdown.
- Do NOT wrap the output in code blocks.
- The first line must be the header.
- Generate 30 rows of realistic farm data.
- Dates must be in YYYY-MM-DD format.
- Use commas as separators.
- Do not include empty rows.

Return ONLY the CSV content.
"""

user_prompt = f"""
I need you to generate a dataset for a bell pepper farm. I need it to make for future reference as well as for training a model.
"""


In [ ]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL).to(device)

In [ ]:
def to_csv(csv_text):
    reader = csv.reader(io.StringIO(csv_text))

    with open("output.csv", "w", newline="") as f:
        writer = csv.writer(f)
        for row in reader:
            writer.writerow(row)

In [ ]:
def generate(quant=True, max_new_tokens=800):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to(device)

    attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=device)

    streamer = TextStreamer(tokenizer)

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        streamer=streamer
    )

    # remove prompt tokens
    res = tokenizer.decode(
        outputs[0][input_ids.shape[-1]:],
        skip_special_tokens=True
    )

    to_csv(res)

    return res

In [ ]:

with gr.Blocks() as ui:
    
    output_box = gr.Textbox(
        label="Output",
        lines=15
    )

    generate_btn = gr.Button("Generate")

    generate_btn.click(
        fn=generate,
        inputs=[],
        outputs=output_box
    )

ui.launch()
